# SO3C scaling study — Kaggle GPURuns the channel and width scaling axes of `so3c_equivariant_set` on canonicaltop tagging. **Set the accelerator to GPU T4 x2** (only one GPU is used — theharness has no DataParallel) and attach the dataset with the three`top_tagging_*.npz` files (see `notebooks/README.md`).Order matters: cell 4 is a **blocking validation** — it must reproduce the CPUnumbers before any scaling run is worth its GPU-hours.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheaderimport torch, os, pathlibprint("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# The repo. Replace REPO_URL/BRANCH if you work from a fork.REPO_URL = "https://github.com/Nervni-Sanya/so33.git"BRANCH   = "feature/so3c-complexification"if not pathlib.Path("so33").exists():    !git clone --depth 1 -b {BRANCH} {REPO_URL} so33%cd so33!pip -q install torchdiffeq

In [ ]:
# Locate the attached dataset (read-only mount). The loader only reads from# cache_dir, so pointing at /kaggle/input works without copying 0.9 GB.import globcands = glob.glob("/kaggle/input/*/top_tagging_train.npz")assert cands, "Attach the dataset with top_tagging_{train,val,test}.npz"DATA = str(pathlib.Path(cands[0]).parent)OUT  = "/kaggle/working/results_scaling"CKPT = "/kaggle/working/ckpt"print("data:", DATA)!ls -la {DATA}

## 1. Blocking validationfloat32 on GPU must reproduce the CPU float64 canonical numbers(**AUC 0.9744, rejection 320**) to within 0.001. If it does not, stop andfix — everything below inherits the error.

In [ ]:
!python -m benchmarks.run_top_tagging     --cache-dir {DATA} --representation constituents --canonical-splits     --epochs 30 --normalize global --seed 0     --device cuda --dtype float32 --batch-size 512     --models so3c_equivariant_set     --results-dir /kaggle/working/results_validate     --ckpt-dir {CKPT} --resume --max-seconds 39000

In [ ]:
import jsonr = json.load(open("/kaggle/working/results_validate/"                   "top_tagging_canonical__so3c_equivariant_set__seed0.json"))auc, rej = r["test_metrics"]["test_auc"], r["test_metrics"]["bg_rej_30"]print(f"GPU float32: AUC {auc:.4f}  rej@0.3 {rej:.0f}   (CPU float64: 0.9744 / 320)")assert abs(auc - 0.9744) < 1e-3, f"PORT BROKEN: AUC {auc:.4f} differs from CPU"print("validation passed — scaling runs are safe to start")

## 2. Channel axis (geometry)`hidden=128`, `act_hidden=32`, channels 4 → 48. This is the axis that grows thegeometric part; the readout width is held fixed on purpose.Each cell is resumable: rerun the same cell after a session restart and itpicks up from the checkpoint and skips finished models.

In [ ]:
CHANNEL_GRID = [4, 8, 16, 32, 48]     # 26k, 39k, 75k, 198k, 387k parametersfor C in CHANNEL_GRID:    bs = 512 if C <= 16 else 256      # (B,K,K) per channel: keep memory bounded    print(f"=== channels={C} ===")    !python -m benchmarks.run_top_tagging         --cache-dir {DATA} --representation constituents --canonical-splits         --epochs 30 --normalize global --seed 0         --device cuda --dtype float32 --batch-size {bs}         --channels {C} --hidden 128 --act-hidden 32         --models so3c_equivariant_set         --results-dir {OUT}/channels_{C}         --ckpt-dir {CKPT}/channels_{C} --resume --max-seconds 39000

## 3. Width axis (generic capacity — control)`channels=4` fixed, readout width 64 → 512. Expectation: saturation. If thiscurve rises as steeply as the channel curve, the "grow the geometry, not theMLP" claim does not hold and the paper must say so.

In [ ]:
WIDTH_GRID = [64, 128, 256, 512]for H in WIDTH_GRID:    print(f"=== hidden={H} ===")    !python -m benchmarks.run_top_tagging         --cache-dir {DATA} --representation constituents --canonical-splits         --epochs 30 --normalize global --seed 0         --device cuda --dtype float32 --batch-size 512         --channels 4 --hidden {H} --act-hidden 32         --models so3c_equivariant_set         --results-dir {OUT}/width_{H}         --ckpt-dir {CKPT}/width_{H} --resume --max-seconds 39000

## 4. Collect

In [ ]:
import json, glob, pathlibrows = []for f in sorted(glob.glob(f"{OUT}/*/*.json")):    r = json.load(open(f))    rows.append((pathlib.Path(f).parent.name, r["n_params"],                 r["test_metrics"]["test_auc"], r["test_metrics"]["bg_rej_30"],                 r["walltime_sec"] / 3600))rows.sort(key=lambda x: x[1])print(f"{'config':<14}{'params':>9}{'AUC':>9}{'rej@0.3':>10}{'hours':>8}")for c, p, a, j, h in rows:    print(f"{c:<14}{p:>9}{a:>9.4f}{j:>10.0f}{h:>8.2f}")# Zip for download; also copy the raw JSONs so figures can be rebuilt locally.!cd /kaggle/working && zip -qr results_scaling.zip results_scaling results_validateprint("download /kaggle/working/results_scaling.zip, unpack into the repo, then:")print("  python -m benchmarks.figure_scaling --results-dir results_scaling")